In [14]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import json, os, re

# ── Ollama ──
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "qwen3.8"          # or mistral, qwen2, phi3 …
OLLAMA_TIMEOUT  = 180

OUTPUT_FILE = "nepali_news.html"

# ── Devanagari Unicode range (U+0900 – U+097F) ──
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')

def has_devanagari(text: str) -> bool:
    return bool(DEVANAGARI_RE.search(text))

print(f"✅ Ollama : {OLLAMA_BASE_URL}  |  Model: {OLLAMA_MODEL}")



✅ Ollama : http://localhost:11434  |  Model: qwen3.8


In [15]:
#Verify Ollama is Alive
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        tags = r.json().get("models", [])
        print(f"🟢 Ollama is running. Available models: {len(tags)}")
        for m in tags:
            print(f"   • {m['name']}")
        return True
    except requests.exceptions.ConnectionError:
        print("🔴 Ollama is NOT running. Start it with: ollama serve")
        return False

check_ollama()


🟢 Ollama is running. Available models: 5
   • qwen3.8:latest
   • gpt-oss:20b
   • qwen2.5-coder:1.5b-base
   • nomic-embed-text:latest
   • llama3.1:8b


True

In [16]:
# Ollama Helper (Chat + Summarize)
def ollama_chat(messages: list, model: str = OLLAMA_MODEL,
                temperature: float = 0.4) -> str:
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 1024},
    }
    r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload,
                      timeout=OLLAMA_TIMEOUT)
    r.raise_for_status()
    return r.json()["message"]["content"].strip()


def ollama_summarize_nepali(articles_text: str) -> str:
    """Ask Ollama for a Devanagari-Nepali summary."""
    messages = [
        {
            "role": "system",
            "content": (
                "तपाईं एक नेपाली समाचार विश्लेषक हुनुहुन्छ। "
                "तपाईंले सधैं देवनागरी लिपिमा नेपालीमा मात्र उत्तर दिनुहुन्छ। "
                "English मा जवाफ दिनु हुँदैन।"
            ),
        },
        {
            "role": "user",
            "content": (
                "तल आजका नेपाली समाचार शीर्षकहरू दिइएको छ। "
                "यी शीर्षकहरूको आधारमा ३-४ वाक्यको सारांश देवनागरी "
                "नेपालीमा लेख्नुहोस्। तटस्थ र तथ्यात्मक रहनुहोस्।\n\n"
                f"{articles_text}"
            ),
        },
    ]
    return ollama_chat(messages)



In [17]:
#Multi Site Scrapper
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ne,nq;q=0.9,en;q=0.5",
}

# ──────────────────────────────────────────────────────
# 1. Sites that are 100 % Devanagari-Nepali
# ──────────────────────────────────────────────────────
SOURCES = [
    ("रेपब्लिक",        "https://republi.co.np/"),
    ("ऑनलाइन खबर",     "https://onlinekhabar.com/"),
    ("सेतोपाटी",        "https://setopati.com/"),
    ("ई-काठमाडौं",     "https://ekantipur.com/"),
    ("भैरवी",           "https://bhairabipost.com/"),
    ("गोरखापत्र",        "https://gorkhapatra.org/"),
    ("न्यूजभिट",        "https://newsvit.com/"),
    ("कतुवा",           "https://kathmandupost.com/"),   # has Nepali section
]

# ──────────────────────────────────────────────────────
# 2. Containers / tags that hold REAL article titles
#    (WordPress & custom themes used by Nepali sites)
# ──────────────────────────────────────────────────────
ARTICLE_SELECTOR = (
    "article, "
    "h2.article-title, h3.article-title, "
    "h2.post-title,   h3.post-title, "
    "h2.entry-title,  h3.entry-title, "
    "h2.news-title,   h3.news-title, "
    "h2.story-title,  h3.story-title, "
    "h2.headline,     h3.headline, "
    ".article-title,  .post-title, "
    ".entry-title,    .news-title, "
    ".story-title,    .headline, "
    ".news-list .item-title, "
    ".featured-articles .title, "
    ".top-news .title, "
    ".home-news a, "
    ".latest-news .title"
)

# ──────────────────────────────────────────────────────
# 3. Elements to EXCLUDE (nav, categories, tags, …)
# ──────────────────────────────────────────────────────
EXCLUDE_SELECTORS = (
    "nav, header, footer, aside, "
    ".navbar, .menu, .sidebar, .widget, "
    ".category, .tag, .breadcrumb, "
    ".pagination, .social, .share, "
    ".comment, .comments, "
    ".author, .profile, "
    ".footer, .header, .banner, "
    ".ad, .advert, .ads, "
    ".related-posts, .related, "
    ".trending-sidebar, .most-read"
)

MIN_TITLE_LEN = 30          # reject short category words
MAX_PER_SITE  = 12


def _is_valid_article_title(text: str) -> bool:
    """Heuristic: is this a real news headline in Devanagari Nepali?"""
    t = text.strip()
    if len(t) < MIN_TITLE_LEN:
        return False
    if not has_devanagari(t):          # must contain Devanagari script
        return False
    # reject things that look like UI / nav text
    bad_words = [
        "लोगिन", "साइन इन", "सुची", "श्रेणी", "ट्याग",
        "टिप्पणी", "साझा", "फ्यासबुक", "ट्विटर",
        "कुनै पनि", "पढ्नुहोस्", "थप", "विस्तृत",
    ]
    for w in bad_words:
        if w in t:
            return False
    return True


def _get_base_url(response_url: str) -> str:
    parts = response_url.rsplit("/", 1)
    return parts[0] if len(parts) > 1 else response_url


def scrape_site(name: str, url: str, max_links: int = MAX_PER_SITE):
    print(f"  📰 {name:.<30s}", end="", flush=True)
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        base = _get_base_url(r.url)

        # --- strip out nav / sidebar / footer etc. first ---
        for sel in EXCLUDE_SELECTORS.split(", "):
            for el in soup.select(sel):
                el.decompose()

        # --- grab candidate <a> tags inside article containers ---
        candidates = []
        for anchor in soup.select(f"{ARTICLE_SELECTOR} a, a {ARTICLE_SELECTOR}"):
            href  = anchor.get("href", "")
            title = anchor.get_text(strip=True)
            if _is_valid_article_title(title) and href:
                full = href if href.startswith("http") else f"{base}{href}"
                candidates.append({"source": name, "title": title, "url": full})

        # --- also try heading tags directly (some sites put title in <h2>/<h3>) ---
        if len(candidates) < 5:
            for heading in soup.select("h2, h3"):
                title = heading.get_text(strip=True)
                a_tag = heading.find("a")
                href  = a_tag["href"] if a_tag else ""
                if _is_valid_article_title(title) and href:
                    full = href if href.startswith("http") else f"{base}{href}"
                    entry = {"source": name, "title": title, "url": full}
                    if entry not in candidates:
                        candidates.append(entry)

        # dedup by URL, keep order
        seen = set()
        unique = []
        for c in candidates:
            if c["url"] not in seen:
                seen.add(c["url"])
                unique.append(c)
            if len(unique) >= max_links:
                break

        print(f"→ {len(unique)} headlines")
        return unique

    except Exception as e:
        print(f"→ ⚠️  {e}")
        return []



In [18]:
#Scrape and Collect 50 Titles
all_articles = []
for name, url in SOURCES:
    all_articles.extend(scrape_site(name, url))

# global de-dup by URL
seen = set()
unique = []
for a in all_articles:
    if a["url"] not in seen and has_devanagari(a["title"]):
        seen.add(a["url"])
        unique.append(a)

TARGET = 50
top_articles = unique[:TARGET]

print(f"\n{'='*60}")
print(f"  Total scraped  : {len(unique)}  (all Devanagari)")
print(f"  Picking top    : {len(top_articles)}")
print(f"{'='*60}\n")

for i, a in enumerate(top_articles, 1):
    print(f"  {i:>2}. [{a['source']}] {a['title'][:72]}")

assert len(top_articles) >= 10, "⚠️  Too few articles — check network / site availability"


  📰 रेपब्लिक......................→ ⚠️  HTTPSConnectionPool(host='republi.co.np', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='republi.co.np', port=443): Failed to resolve 'republi.co.np' ([Errno 11002] getaddrinfo failed)"))
  📰 ऑनलाइन खबर....................→ 12 headlines
  📰 सेतोपाटी......................→ ⚠️  HTTPSConnectionPool(host='setopati.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'setopati.com'. (_ssl.c:1016)")))
  📰 ई-काठमाडौं....................→ 12 headlines
  📰 भैरवी.........................→ ⚠️  HTTPSConnectionPool(host='bhairabipost.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='bhairabipost.com', port=443): Failed to resolve 'bhairabipost.com' ([Errno 11001] getaddrinfo failed)"))
  📰 गोरखापत्र......

In [19]:
# Summarize via Ollama
# Build a compact digest string
digest_lines = [f"{i}. {a['title']}" for i, a in enumerate(top_articles, 1)]
digest_text  = "\n".join(digest_lines)

print("🤖 Ollama ले देवनागरी सारांश लेखिरहेको छ …\n")
try:
    ai_summary = ollama_summarize_nepali(digest_text)
    print("✅ AI सारांश:\n")
    print(ai_summary)
except Exception as e:
    ai_summary = f"(Ollama कल विफल: {e})"
    print(ai_summary)


🤖 Ollama ले देवनागरी सारांश लेखिरहेको छ …

✅ AI सारांश:




In [9]:
#Generate Hyler linked html page
def build_html(articles, ai_summary="", output=OUTPUT_FILE):
    today = datetime.now().strftime("%Y/%m/%d")

    grouped = {}
    for a in articles:
        grouped.setdefault(a["source"], []).append(a)

    rows_html = ""
    num = 0
    for source, items in grouped.items():
        rows_html += f'\n    <div class="source-block">'
        rows_html += f'<h2>📰 {source}</h2>\n    <ol start="{num+1}">'
        for a in items:
            num += 1
            # escape HTML entities in title
            safe_title = (a["title"]
                          .replace("&", "&amp;")
                          .replace("<", "&lt;")
                          .replace(">", "&gt;")
                          .replace('"', "&quot;"))
            rows_html += (
                f'      <li>\n'
                f'        <a href="{a["url"]}" target="_blank" rel="noopener">'
                f'{safe_title}</a>\n'
                f'      </li>'
            )
        rows_html += "\n    </ol>\n    </div>"

    summary_block = ""
    if ai_summary and "विफल" not in ai_summary:
        safe_sum = (ai_summary
                    .replace("&", "&amp;")
                    .replace("<", "&lt;")
                    .replace(">", "&gt;"))
        summary_block = f"""
    <div class="ai-summary">
      <h2>🤖 AI दैनिक सारांश  <span class="badge">{OLLAMA_MODEL}</span></h2>
      <p>{safe_sum}</p>
    </div>"""

    html = f"""<!DOCTYPE html>
<html lang="ne">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>नेपाल समाचार — {today}</title>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Mukta:wght@400;600;700&display=swap');
    :root {{
      --bg      : #0e1117;
      --card    : #181c25;
      --accent  : #d4393e;
      --text    : #dce0e8;
      --link    : #5eb1ef;
      --link-hv : #90caf9;
      --muted   : #7a7f8a;
      --border  : #252a36;
    }}
    * {{ box-sizing:border-box; margin:0; padding:0; }}
    body {{
      font-family:'Mukta','Noto Sans Devanagari',sans-serif;
      background:var(--bg); color:var(--text);
      line-height:1.7; padding:2rem 1rem;
    }}
    .container {{ max-width:840px; margin:0 auto; }}
    header {{
      text-align:center; margin-bottom:2rem;
      padding-bottom:1.4rem; border-bottom:2px solid var(--accent);
    }}
    header h1 {{ font-size:2rem; color:#fff; }}
    header p  {{ color:var(--muted); margin-top:.35rem; font-size:.92rem; }}
    .ai-summary {{
      background:linear-gradient(135deg,#162030,#1a2640);
      border-left:4px solid var(--link);
      border-radius:8px; padding:1.2rem 1.5rem; margin-bottom:1.8rem;
    }}
    .ai-summary h2 {{ font-size:1.05rem; color:var(--link); margin-bottom:.5rem; }}
    .ai-summary p  {{ font-size:.95rem; color:#cdd3de; line-height:1.8; }}
    .source-block {{
      background:var(--card); border-radius:10px;
      padding:1.1rem 1.4rem; margin-bottom:1.1rem;
      border:1px solid var(--border);
    }}
    .source-block h2 {{ font-size:1rem; color:var(--accent); margin-bottom:.6rem; }}
    ol {{ padding-left:1.4rem; }}
    li {{ padding:.32rem 0; border-bottom:1px solid var(--border); }}
    li:last-child {{ border-bottom:none; }}
    a {{ color:var(--link); text-decoration:none; }}
    a:hover {{ color:var(--link-hv); text-decoration:underline; }}
    .badge {{
      display:inline-block; background:var(--accent); color:#fff;
      font-size:.68rem; padding:.1rem .5rem; border-radius:10px;
      margin-left:.4rem; vertical-align:middle;
    }}
    footer {{ text-align:center; margin-top:2rem; color:var(--muted); font-size:.78rem; }}
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>🇳🇵 नेपाल समाचार — {today}</h1>
      <p>शीर्ष {len(articles)} समाचार &middot; {len(grouped)} स्रोत &middot; Python + Ollama द्वारा तयार पारिएको</p>
    </header>
    {summary_block}
    {rows_html}
    <footer>
      {datetime.now().strftime("%H:%M:%S")} मा एकत्र गरिएको
      &middot; Ollama ({OLLAMA_MODEL})
    </footer>
  </div>
</body>
</html>"""

    with open(output, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"\n✅ HTML सेभ भयो → {os.path.abspath(output)}")
    return output


html_path = build_html(top_articles, ai_summary=ai_summary)




✅ HTML saved → C:\Users\kusha\AgenticAIWs\nepali_news.html
   Open it in your browser:
   file://nepali_news.html


In [10]:
#Inline preview + JSON
try:
    from IPython.display import HTML, display
    with open(html_path, encoding="utf-8") as f:
        display(HTML(f.read()))
except Exception:
    print(f"ब्राउजरमा खोल्नुहोस्: file://{html_path}")

json_path = "nepali_news.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({
        "date":       datetime.now().isoformat(),
        "model":      OLLAMA_MODEL,
        "ai_summary": ai_summary,
        "articles":   top_articles,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ JSON → {os.path.abspath(json_path)}")



In [11]:
#Json Export
# json_path = "nepali_news.json"
# with open(json_path, "w", encoding="utf-8") as f:
#     json.dump({
#         "date": datetime.now().isoformat(),
#         "model": OLLAMA_MODEL,
#         "ai_summary": ai_summary,
#         "articles": top_articles,
#     }, f, ensure_ascii=False, indent=2)
# print(f"✅ JSON saved → {os.path.abspath(json_path)}")


✅ JSON saved → C:\Users\kusha\AgenticAIWs\nepali_news.json
